<a href="https://colab.research.google.com/github/pangpntt/order-etl-pipeline/blob/main/data_quality_checker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Project")
DATA_DIR = PROJECT_DIR / "data"
REPORT_DIR = PROJECT_DIR / "reports"
REPORT_DIR.mkdir(exist_ok=True)

for f in sorted(DATA_DIR.glob("*.csv")):
    print(f.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_orders_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


In [5]:
orders= pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Project/data/olist_orders_dataset.csv",
                    parse_dates=["order_purchase_timestamp", "order_delivered_customer_date"])


In [6]:
items = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Project/data/olist_order_items_dataset.csv")

In [7]:
customers = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Project/data/olist_customers_dataset.csv")

In [8]:
for name, df in [("orders", orders), ("items", items), ("customers", customers)]:
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

orders: 99,441 rows, 8 columns
items: 112,650 rows, 7 columns
customers: 99,441 rows, 5 columns


In [7]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [8]:
items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [11]:
results = []

def record(check, table, bad_rows):
    results.append({"check": check, "table": table, "issues": len(bad_rows)})
    if len(bad_rows) > 0:
        bad_rows.to_csv(REPORT_DIR / f"issues_{check}_{table}.csv", index=False)

In [12]:
### Check null columns

key_columns = {
    "orders": (orders, ["order_id", "customer_id"]),
    "order_items": (items, ["order_id", "product_id", "price"]),
    "customers": (customers, ["customer_id"]),
}

for table, (df, cols) in key_columns.items():
    bad_rows = df[df[cols].isna().any(axis=1)]
    record("null_key_columns", table, bad_rows)
    print(f"{table}: {len(bad_rows)} rows with null in {cols}")

orders: 0 rows with null in ['order_id', 'customer_id']
order_items: 0 rows with null in ['order_id', 'product_id', 'price']
customers: 0 rows with null in ['customer_id']


In [13]:
### Check duplicate

duplicate_checks = {
    "orders": (orders, ["order_id"]),
    "order_items": (items, ["order_id", "order_item_id"]),
    "customers": (customers, ["customer_id"]),
}

for table, (df, key) in duplicate_checks.items():
    bad_rows = df[df.duplicated(subset=key, keep=False)]
    record("duplicate_keys", table, bad_rows)
    print(f"{table}: {len(bad_rows)} rows with duplicate {key}")

orders: 0 rows with duplicate ['order_id']
order_items: 0 rows with duplicate ['order_id', 'order_item_id']
customers: 0 rows with duplicate ['customer_id']


In [14]:
### Check orphan records

orphan_orders = orders[~orders["customer_id"].isin(customers["customer_id"])]
record("orphan_records", "orders", orphan_orders)
print(f"orders without a matching customer: {len(orphan_orders)}")

orphan_items = items[~items["order_id"].isin(orders["order_id"])]
record("orphan_records", "order_items", orphan_items)
print(f"order_items without a matching order: {len(orphan_items)}")

orders without a matching customer: 0
order_items without a matching order: 0


In [15]:
### Check negative amounts

negative_amounts = items[(items["price"] < 0) | (items["freight_value"] < 0)]
record("negative_amounts", "order_items", negative_amounts)
print(f"order_items with negative price or freight: {len(negative_amounts)}")

order_items with negative price or freight: 0


In [16]:
### Check delivered before purchase

delivered_before_purchase = orders[
    orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]
]
record("delivered_before_purchase", "orders", delivered_before_purchase)
print(f"orders delivered before purchase date: {len(delivered_before_purchase)}")

orders delivered before purchase date: 0


In [17]:
### Check delivered without a delivery date

delivered_without_date = orders[
    (orders["order_status"] == "delivered")
    & orders["order_delivered_customer_date"].isna()
]
record("delivered_without_date", "orders", delivered_without_date)
print(f"delivered orders without delivery date: {len(delivered_without_date)}")

delivered orders without delivery date: 8


In [18]:
delivered_without_date

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
3002,2d1e2d5bf4dc7227b3bfebb81328c15f,ec05a6d8558c6455f0cbbd8a420ad34f,delivered,2017-11-28 17:44:07,2017-11-28 17:56:40,2017-11-30 18:12:23,NaT,2017-12-18 00:00:00
20618,f5dd62b788049ad9fc0526e3ad11a097,5e89028e024b381dc84a13a3570decb4,delivered,2018-06-20 06:58:43,2018-06-20 07:19:05,2018-06-25 08:05:00,NaT,2018-07-16 00:00:00
43834,2ebdfc4f15f23b91474edf87475f108e,29f0540231702fda0cfdee0a310f11aa,delivered,2018-07-01 17:05:11,2018-07-01 17:15:12,2018-07-03 13:57:00,NaT,2018-07-30 00:00:00
79263,e69f75a717d64fc5ecdfae42b2e8e086,cfda40ca8dd0a5d486a9635b611b398a,delivered,2018-07-01 22:05:55,2018-07-01 22:15:14,2018-07-03 13:57:00,NaT,2018-07-30 00:00:00
82868,0d3268bad9b086af767785e3f0fc0133,4f1d63d35fb7c8999853b2699f5c7649,delivered,2018-07-01 21:14:02,2018-07-01 21:29:54,2018-07-03 09:28:00,NaT,2018-07-24 00:00:00
92643,2d858f451373b04fb5c984a1cc2defaf,e08caf668d499a6d643dafd7c5cc498a,delivered,2017-05-25 23:22:43,2017-05-25 23:30:16,NaN,NaT,2017-06-23 00:00:00
97647,ab7c89dc1bf4a1ead9d6ec1ec8968a84,dd1b84a7286eb4524d52af4256c0ba24,delivered,2018-06-08 12:09:39,2018-06-08 12:36:39,2018-06-12 14:10:00,NaT,2018-06-26 00:00:00
98038,20edc82cf5400ce95e1afacc25798b31,28c37425f1127d887d7337f284080a0f,delivered,2018-06-27 16:09:12,2018-06-27 16:29:30,2018-07-03 19:26:00,NaT,2018-07-19 00:00:00


In [19]:
report = pd.DataFrame(results)
report["status"] = report["issues"].apply(lambda n: "PASS" if n == 0 else "FAIL")
report.to_csv(REPORT_DIR / "quality_report.csv", index=False)

print(report.to_string(index=False))
print(f"\nTotal issues found: {report['issues'].sum()}")

                    check       table  issues status
         null_key_columns      orders       0   PASS
         null_key_columns order_items       0   PASS
         null_key_columns   customers       0   PASS
           duplicate_keys      orders       0   PASS
           duplicate_keys order_items       0   PASS
           duplicate_keys   customers       0   PASS
           orphan_records      orders       0   PASS
           orphan_records order_items       0   PASS
         negative_amounts order_items       0   PASS
delivered_before_purchase      orders       0   PASS
   delivered_without_date      orders       8   FAIL

Total issues found: 8


In [20]:
bad_order_ids = set(delivered_without_date["order_id"])

orders_clean = orders[~orders["order_id"].isin(bad_order_ids)].copy()
orders_rejected = orders[orders["order_id"].isin(bad_order_ids)].copy()

print(f"orders: {len(orders):,}")
print(f"clean: {len(orders_clean):,}")
print(f"rejected: {len(orders_rejected):,}")

orders: 99,441
clean: 99,433
rejected: 8


In [21]:
order_totals = (
    items.groupby("order_id")
    .agg(
        item_count=("order_item_id", "count"),
        items_total=("price", "sum"),
        freight_total=("freight_value", "sum"),
    )
    .reset_index()
)

fact_orders = (
    orders_clean
    .merge(customers[["customer_id", "customer_city", "customer_state"]],
           on="customer_id", how="left")
    .merge(order_totals, on="order_id", how="left")
)

fact_orders["delivery_days"] = (
    fact_orders["order_delivered_customer_date"]
    - fact_orders["order_purchase_timestamp"]
).dt.days

assert len(fact_orders) == len(orders_clean), "Row count changed after join!"

print(fact_orders.shape)
fact_orders.head()

(99433, 14)


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_city,customer_state,item_count,items_total,freight_total,delivery_days
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,sao paulo,SP,1.0,29.99,8.72,8.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,barreiras,BA,1.0,118.70,22.76,13.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,vianopolis,GO,1.0,159.90,19.22,9.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,sao goncalo do amarante,RN,1.0,45.00,27.20,13.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,santo andre,SP,1.0,19.90,8.72,2.0


In [22]:
import sqlite3

In [23]:
DB_PATH = PROJECT_DIR / "olist.db"

conn = sqlite3.connect(DB_PATH)

fact_orders.to_sql("fact_orders", conn, if_exists="replace", index=False)
orders_rejected.to_sql("orders_rejected", conn, if_exists="replace", index=False)

loaded = pd.read_sql("SELECT COUNT(*) AS n FROM fact_orders", conn)["n"][0]
assert loaded == len(fact_orders), "Loaded row count does not match!"
print(f"fact_orders loaded: {loaded:,} rows")

conn.close()

fact_orders loaded: 99,433 rows


In [24]:
conn = sqlite3.connect(DB_PATH)

query = """
SELECT customer_state,
       COUNT(*) AS orders,
       ROUND(AVG(delivery_days), 1) AS avg_delivery_days
FROM fact_orders
WHERE delivery_days IS NOT NULL
GROUP BY customer_state
ORDER BY avg_delivery_days DESC
LIMIT 5
"""

pd.read_sql(query, conn)

,customer_state,orders,avg_delivery_days
0,RR,41,29.0
1,AP,67,26.7
2,AM,145,26.0
3,AL,397,24.0
4,PA,946,23.3
